In [3]:
from MACS3.IO.Parser import FragParser
from MACS3.IO.BedGraphIO import bedGraphIO
from MACS3.Utilities.Logger import logging
import numpy as np
from scipy import sparse
import anndata as ad
# we can use the pre-configured logging function from MACS3 to monitor memory usage...
logger = logging.getLogger("demo")
info = logger.info
# Note, you can replace `print` with `info` to show the time and memory usage at that moment.
from MACS3.Signal.PairedEndTrack import PETrackII
from MACS3.Signal.Region import Regions
import pandas as pd
import anndata as ad
from ncls import NCLS

In [ ]:
# simulate a toy data

petrack = PETrackII()
petrack.add_loc(b"chr1", 0, 100, barcode=b"A", count=2)   # peak_1
petrack.add_loc(b"chr1", 70, 270, barcode=b"A", count=1)   # peak_1&2
petrack.add_loc(b"chr1", 0, 100, barcode=b"B", count=3)   # peak_1
petrack.add_loc(b"chr1", 175, 325, barcode=b"C", count=4)   # peak_2
petrack.finalize()

regions = Regions()
regions.add_loc(b"chr1", 0, 100)    # peak_1 A:3 B:3 C:0
regions.add_loc(b"chr1", 200, 300)  # peak_2 A:1 B:0 C:4
regions.add_loc(b"chr1", 500, 600)  # peak_3 A:0 B:0 C:0

In [48]:

def np_sort_search(petrack,regions):
        # barcode mapping
        barcode_items = sorted(petrack.barcode_dict.items(), key=lambda x: x[1])
        barcodes = [b.decode() if isinstance(b, (bytes, bytearray)) else str(b) for b, _ in barcode_items]
        barcode_id_to_row = {barcode_id: i for i, (_, barcode_id) in enumerate(barcode_items)}
        n_cells = len(barcodes)

        peak_names = []
        peak_data = []
        rows = []
        columns = []
        data = []

        regions.sort()
        peak_counter = 0

        for chrom in sorted(regions.regions.keys()):
            if chrom not in petrack.locations:
                continue

            regions_c = regions.regions[chrom]
            if not regions_c:
                continue

            local_peak_indices = []
            for (start, end) in regions_c:
                peak_counter += 1
                peak_names.append(f"peak_{peak_counter}")
                peak_data.append((chrom.decode() if isinstance(chrom, (bytes, bytearray)) else str(chrom), int(start), int(end)))
                local_peak_indices.append(peak_counter - 1)

            peak_starts = np.array([p[0] for p in regions_c], dtype=np.int64)
            peak_ends = np.array([p[1] for p in regions_c], dtype=np.int64)
            local_peak_indices = np.array(local_peak_indices, dtype=np.int64)

            fragment_locs = petrack.locations[chrom]
            fragment_barcodes = petrack.barcodes[chrom]
            if len(fragment_locs) == 0:
                continue

            for frag_idx, frag in enumerate(fragment_locs):
                frag_start = frag[0]
                frag_end = frag[1]
                bc_id = int(fragment_barcodes[frag_idx])
                row_id = barcode_id_to_row.get(bc_id, -1)
                if row_id < 0:
                    continue

                j = np.searchsorted(peak_starts, frag_start, side='left')
                while j < peak_starts.size and peak_starts[j] <= frag_end:
                    if peak_ends[j] <= frag_end:
                        rows.append(row_id)
                        columns.append(int(local_peak_indices[j]))
                        data.append(int(frag[2]))
                    j += 1
        X = sparse.coo_matrix((data, (rows, columns)), shape=(n_cells, peak_counter), dtype=np.int32).tocsr()
        obs = pd.DataFrame(index=barcodes)
        var = pd.DataFrame(peak_data, columns=['chrom', 'start', 'end'], index=peak_names)
        adata_peaks = ad.AnnData(X, obs=obs, var=var)
        return adata_peaks


In [49]:
adata_np_ss = np_sort_search(petrack,regions)
print(adata_np_ss)
print(adata_np_ss.X.toarray())

AnnData object with n_obs × n_vars = 3 × 3
    var: 'chrom', 'start', 'end'
[[2 0 0]
 [3 0 0]
 [0 4 0]]


In [51]:
# ncls implementation
def build_anndata_ncls(petrack, regions, binary=False):
        regions.sort()

        # Barcode mapping
        barcode_items = sorted(petrack.barcode_dict.items(), key=lambda x: x[1])
        barcodes = [b.decode() if isinstance(b, (bytes, bytearray)) else str(b) for b, _ in barcode_items]
        barcode_ids = np.array([i for _, i in barcode_items], dtype=np.int64)

        n_cells = len(barcodes)
        if n_cells == 0:
            return ad.AnnData(X=sparse.csr_matrix((0, 0), dtype=np.int32))

        id_map = np.full(barcode_ids.max() + 1, -1, dtype=np.int64)
        id_map[barcode_ids] = np.arange(n_cells, dtype=np.int64)

        # Outputs
        peak_names = []
        peak_data = []
        rows = []
        cols = []
        data = []
        peak_counter = 0

        for chrom in sorted(regions.regions.keys()):
            if chrom not in petrack.locations:
                continue

            regions_c = regions.regions[chrom]
            if not regions_c:
                continue

            # register peaks (global)
            local_peak_indices = []
            chrom_str = chrom.decode() if isinstance(chrom, (bytes, bytearray)) else str(chrom)
            for (start, end) in regions_c:
                peak_counter += 1
                peak_names.append(f"peak_{peak_counter}")
                peak_data.append((chrom_str, int(start), int(end)))
                local_peak_indices.append(peak_counter - 1)

            # fragments on this chromosome
            fragment_locs = petrack.locations[chrom][:petrack.size[chrom]]
            fragment_barcodes = petrack.barcodes[chrom][:petrack.size[chrom]]
            if len(fragment_locs) == 0:
                continue

            frag_starts = fragment_locs['l'].astype(np.int64)
            frag_ends = fragment_locs['r'].astype(np.int64)
            frag_counts = fragment_locs['c'].astype(np.int64)
            frag_bc_ids = fragment_barcodes.astype(np.int64)

            frag_rows = id_map[frag_bc_ids]  # barcode -> row index

            idx = np.arange(len(frag_starts), dtype=np.int64)
            ncls = NCLS(frag_starts, frag_ends, idx)

            # Query peaks: peak must be inside fragment
            for peak_idx, (peak_start, peak_end) in enumerate(regions_c):
                for frag_start, frag_end, frag_i in ncls.find_overlap(peak_start, peak_end):
            # enforce peak inside fragment
                    if frag_start <= peak_start and peak_end <= frag_end:
                        row_id = frag_rows[frag_i]
                        if row_id >= 0:
                            rows.append(row_id)
                            cols.append(local_peak_indices[peak_idx])
                            data.append(int(frag_counts[frag_i]))
                            
        X = sparse.coo_matrix((data, (rows, cols)),
                            shape=(n_cells, peak_counter),
                            dtype=np.int32).tocsr()

        obs = pd.DataFrame(index=barcodes)
        var = pd.DataFrame(peak_data, columns=["chrom", "start", "end"], index=peak_names)
        return ad.AnnData(X=X, obs=obs, var=var)

In [52]:
adata_ncls = build_anndata_ncls(petrack,regions)
print(adata_ncls)
print(adata_ncls.X.toarray())

AnnData object with n_obs × n_vars = 3 × 3
    var: 'chrom', 'start', 'end'
[[2 0 0]
 [3 0 0]
 [0 4 0]]


In [ ]:
# two pointer sweep like exclude

def two_pointer_sweep(petrack,regions):
    barcode_items = list(petrack.barcode_dict.items())
    barcodes = [b.decode() if isinstance(b, (bytes, bytearray)) else str(b) for b, _ in barcode_items]
    barcode_ids = np.array([i for _, i in barcode_items], dtype=np.int64)

    id_map = np.full(barcode_ids.max() + 1, -1, dtype=np.int64)
    id_map[barcode_ids] = np.arange(len(barcode_ids), dtype=np.int64)
    n_cells = len(barcodes)

    # # array of locataions and counts
    # fragment_locs = petrack.locations[chrom][:petrack.size[chrom]]
    # # arrray of barcodes
    # fragment_barcodes = petrack.barcodes[chrom][:petrack.size[chrom]]

    peak_names = []
    peak_data = []

    rows = [] #barcodes index
    columns = [] #peaks index
    data = [] # count data int

    barcode_items = sorted(petrack.barcode_dict.items(), key=lambda x: x[1])
    barcodes = [b.decode() if isinstance(b, (bytes, bytearray)) else str(b) for b, _ in barcode_items]
    barcode_id_to_row = {barcode_id: i for i, (_, barcode_id) in enumerate(barcode_items)}
    n_cells = len(barcodes)

    regions.sort()
    peak_counter = 0
    for chrom in regions.regions.keys():
        size = petrack.size[chrom]
        locs = petrack.locations[chrom]
        #barcodes = petrack.barcodes
        regions_c = regions.regions[chrom]
        local_peak = []
        ### regions empty skip
        for (start, end) in regions_c:
            peak_counter += 1
            peak_names.append(f"peak_{peak_counter}")
            peak_data.append((chrom.decode(),start,end))
            local_peak.append(peak_counter - 1)

        fragment_locs = petrack.locations[chrom][:petrack.size[chrom]]
        fragment_barcodes = petrack.barcodes[chrom][:petrack.size[chrom]]

        if len(fragment_locs) == 0:
            continue

        frag_idx = 0
        local_peak_idx = 0
        frag_len = len(fragment_locs)
        peak_len = len(regions_c)
        frag = fragment_locs[frag_idx]
        remaining_frag_len = frag_len - 1
        peak = regions_c[local_peak_idx] # (start, end)
        remaining_peak_len = peak_len - 1

        # inside two_pointer_sweep, replace only the while-loop block with this
        back_trace_frag = 0
        while True:
            frag_start, frag_end = frag[0], frag[1]
            peak_start, peak_end = peak[0], peak[1]

            # peak overlap fragment
            if frag_start <= peak_end and peak_start <= frag_end:
                bc_id = int(fragment_barcodes[frag_idx])
                row_id = barcode_id_to_row.get(bc_id, -1)
                if row_id >= 0:
                    rows.append(int(row_id))
                    columns.append(local_peak[local_peak_idx])
                    data.append(int(frag[2]))
                    #print(f"peak {local_peak_idx} overlaps with fragment {frag_idx}, row_id: {row_id}, count: {frag[2]}")

                if frag_end > peak_end:
                    # overhang case, perhaps the frag can still contain next peak(s)
                    back_trace_frag += 1
                remaining_frag_len -= 1
                if remaining_frag_len >= 0:
                    frag_idx += 1
                    frag = fragment_locs[frag_idx]
                    continue
                else:
                    break

            # if fragment ends before peak ends -> advance fragment
            if frag_end < peak_end:
                remaining_frag_len -= 1
                if remaining_frag_len >= 0:
                    frag_idx += 1
                    frag = fragment_locs[frag_idx]
                else:
                    break
            else:
                # advance peak
                remaining_peak_len -= 1
                if remaining_peak_len >= 0:
                    local_peak_idx += 1
                    peak = regions_c[local_peak_idx]
                    # we will also check if backtrace > 0 or not, if so, we need to move back the fragment pointer to check those peaks that were skipped due to overhang
                    if back_trace_frag > 0:
                        frag_idx -= back_trace_frag
                        remaining_frag_len += back_trace_frag
                        frag = fragment_locs[frag_idx]
                    back_trace_frag = 0
                else:
                    break
        #print(f"remaining fragments: {remaining_frag_len}, remaining peaks: {remaining_peak_len}")

    n_peaks = peak_counter
    x = sparse.coo_matrix((data,(rows,columns)),shape=(n_cells,n_peaks)).tocsr()
    obs = pd.DataFrame(index=barcodes)
    #obs['n_fragments_in_peaks'] = np.asarray(x.sum(axis=1)).ravel()
    var = pd.DataFrame(peak_data, columns=['chrom', 'start', 'end'], index=peak_names)

    adata_peaks_loop = ad.AnnData(X=x, obs=obs, var=var)
    return adata_peaks_loop


In [54]:
adata_two = two_pointer_sweep(petrack,regions)
print(adata_two)
print(adata_two.X.toarray())

peak 0 overlaps with fragment 0, row_id: 0, count: 2
peak 0 overlaps with fragment 1, row_id: 1, count: 3
peak 0 overlaps with fragment 2, row_id: 0, count: 1
peak 1 overlaps with fragment 2, row_id: 0, count: 1
peak 1 overlaps with fragment 3, row_id: 2, count: 4
remaining fragments: -1, remaining peaks: 1
AnnData object with n_obs × n_vars = 3 × 3
    var: 'chrom', 'start', 'end'
[[3 1 0]
 [3 0 0]
 [0 4 0]]


In [17]:
## test andata from package

a = petrack.return_anndata(regions, method = "ncls")
a.X.toarray()


array([[2, 0, 0],
       [3, 0, 0],
       [0, 4, 0]], dtype=int32)